# 04 Models Contract And Entry Points

`OctoSense.models` is the module that provides OctoSense's
`Model Repository` and shared model boundary. It solves the
problem that wireless sensing baselines often need to be
reimplemented and adapted again for slightly different input
layouts. The design uses `octo.models.load(...)`, `ModelHandle`,
and public pipeline helpers so model-specific adaptation stays
inside the framework while training and evaluation code can stay
consistent across models. This notebook shows that result by taking
one shared dataset view, applying explicit `entry_overrides`, and
following the public path into model input.

### Set Up Model-Boundary Inspection Helpers

Import the tensor and metadata helpers required to construct one
small shared dataset that can be reused across multiple public
model-entry contracts.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = (NOTEBOOK_DIR / "../..").resolve()
SRC_ROOT = REPO_ROOT / "src"
TEST_DATA_ROOT = NOTEBOOK_DIR / "data"
PRESET_ROOT = NOTEBOOK_DIR / "presets"
NOTEBOOK_TREE_DEPTH = 2
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from loguru import logger
import octosense as octo
import torch
from octosense.io.semantics.metadata import SignalMetadata


logger.remove()
logger.add(sys.stderr, colorize=False, format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {message}")
logger.info("Notebook setup completed with repository-relative paths")
logger.info("Default describe tree render depth set to {}", NOTEBOOK_TREE_DEPTH)

### Inspect Public Model Handles And Their Input Contracts

Resolve several built-in model handles, print their contract
summaries, and keep `entry_overrides` explicit so the boundary
remains inspectable before any pipeline wiring happens.

In [ ]:
logger.info("Resolving model handles first so the boundary stays explicit")
torch.manual_seed(7)
resnet_handle = octo.models.load("resnet18", entry_overrides={"num_classes": 3})
rfnet_handle = octo.models.load("rfnet", entry_overrides={"num_classes": 3})
mlp_handle = octo.models.load("mlp", entry_overrides={"num_classes": 3})

shared_demo_tensor = torch.randn(8, 16, 12, 2)
shared_demo_labels = [0, 1, 0, 1, 0, 1, 0, 1]
shared_demo_dataset = octo.datasets.from_tensor(
    shared_demo_tensor,
    shared_demo_labels,
    axis_names=("sample", "time", "subc", "rx"),
    modalities=["wifi"],
    metadata=SignalMetadata(
        sample_rate=1000.0,
        center_freq=5.18e9,
        bandwidth=20e6,
        modality="WiFi_CSI",
        reader_id="NotebookModelBoundaryReader-v1",
        capture_device="NotebookModelBoundaryDevice",
    ),
    dataset_id="demo-models-shared",
    variant="contract-preview",
)
shared_sample = shared_demo_dataset.get_split("test")[0][0]
logger.info("shared demo batch shape: {}", tuple(shared_demo_tensor.shape))
logger.info("shared demo sample shape: {}", tuple(shared_sample.shape))
logger.info("resnet18 input contract: {}", resnet_handle.get_input_contract())
logger.info("rfnet input contract: {}", rfnet_handle.get_input_contract())
logger.info("mlp input contract: {}", mlp_handle.get_input_contract())
logger.info("rfnet unresolved entry_overrides: {}", rfnet_handle.unresolved_entry_overrides)
logger.info("The runnable task lane below reuses this same dataset for resnet18 and rfnet with num_classes=2")

### Drive Model Entry From Dataset Semantics Via AutoAlign

Reuse one small shared WiFi dataset, wire that same dataset into
multiple `octo.pipelines.load(...)` model entries, and confirm that
each `auto_align` hint resolves a different runnable shape for the
target model boundary.

In [ ]:
logger.info("Driving the same shared dataset through resnet18 and rfnet via model-specific AutoAlign")
resnet_pipeline = octo.pipelines.load(
    task_id="classification/gesture",
    task_binding="gesture",
    model_id="resnet18",
    dataset=shared_demo_dataset,
    entry_overrides={"num_classes": 2},
)
rfnet_pipeline = octo.pipelines.load(
    task_id="classification/gesture",
    task_binding="gesture",
    model_id="rfnet",
    dataset=shared_demo_dataset,
    entry_overrides={"num_classes": 2},
)
resnet_pipeline.auto_align({"channel": "rx", "height": "time", "width": "subc"})
rfnet_pipeline.auto_align({"time": "time", "feature": "subc*rx"})

resnet_entry_sample = resnet_pipeline.transform(shared_sample)
rfnet_entry_sample = rfnet_pipeline.transform(shared_sample)
resnet_logits = resnet_pipeline.infer(
    shared_sample,
    device="cpu",
    seed=7,
    batch_size=4,
    num_workers=0,
)
rfnet_logits = rfnet_pipeline.infer(
    shared_sample,
    device="cpu",
    seed=7,
    batch_size=4,
    num_workers=0,
)
logger.info("shared dataset batch shape reused by both pipelines: {}", tuple(shared_demo_tensor.shape))
logger.info("shared sample shape reused by both pipelines: {}", tuple(shared_sample.shape))
logger.info("resnet18 aligned entry shape: {}", tuple(resnet_entry_sample.shape))
logger.info("rfnet aligned entry shape: {}", tuple(rfnet_entry_sample.shape))
logger.info("resnet18 infer logits shape: {}", tuple(resnet_logits.shape))
logger.info("rfnet infer logits shape: {}", tuple(rfnet_logits.shape))